# Image2GPS Baseline Data and Dataloaders

This notebook shows how to use the train/validation data from `01_data_collection_preprocessing.ipynb`:

```text
data/image2gps_dataset/train/metadata.csv
data/image2gps_dataset/validation/metadata.csv
```

We are not using a local test split for now because the dataset is small and the professor leaderboard/test set is the true held-out test. The notebook follows the example `Release_baseline_model.ipynb` style by creating a custom dataset class and dataloaders before any model code.


In [ ]:
# Uncomment and run this cell if your kernel is missing dependencies.
# %pip install datasets torch torchvision numpy pillow matplotlib

# Data

## Loading the Train and Validation Datasets

The baseline loads a Hugging Face dataset with `load_dataset(...)`. For our local files, we use Hugging Face's built-in `imagefolder` loader. It reads each split folder and its `metadata.csv` file.


In [ ]:
import os

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
PATH_TO_YOUR_DATA_FOLDER = os.path.join(REPO_ROOT, "data", "image2gps_dataset")

print(PATH_TO_YOUR_DATA_FOLDER)

In [ ]:
from datasets import load_dataset, Image

dataset_train = load_dataset("imagefolder", data_dir=PATH_TO_YOUR_DATA_FOLDER, split="train")
dataset_validation = load_dataset("imagefolder", data_dir=PATH_TO_YOUR_DATA_FOLDER, split="validation")


In [ ]:
print(dataset_train)
print(dataset_validation)


In [ ]:
dataset_train[0]

## Defining the Custom Dataset Class

This is the same `GPSImageDataset` pattern from the baseline notebook.

In [ ]:
import torch
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset
import numpy as np


class GPSImageDataset(Dataset):
    def __init__(self, hf_dataset, transform=None, lat_mean=None, lat_std=None, lon_mean=None, lon_std=None):
        self.hf_dataset = hf_dataset
        self.transform = transform

        # Compute mean and std from the dataframe if not provided
        self.latitude_mean = lat_mean if lat_mean is not None else np.mean(np.array(self.hf_dataset['Latitude']))
        self.latitude_std = lat_std if lat_std is not None else np.std(np.array(self.hf_dataset['Latitude']))
        self.longitude_mean = lon_mean if lon_mean is not None else np.mean(np.array(self.hf_dataset['Longitude']))
        self.longitude_std = lon_std if lon_std is not None else np.std(np.array(self.hf_dataset['Longitude']))

    def __len__(self):
        return len(self.hf_dataset)

    def __getitem__(self, idx):
        # Extract data
        example = self.hf_dataset[idx]

        # Load and process the image
        image = example['image']
        latitude = example['Latitude']
        longitude = example['Longitude']
        # image = image.rotate(-90, expand=True)
        if self.transform:
            image = self.transform(image)

        # Normalize GPS coordinates
        latitude = (latitude - self.latitude_mean) / self.latitude_std
        longitude = (longitude - self.longitude_mean) / self.longitude_std
        gps_coords = torch.tensor([latitude, longitude], dtype=torch.float32)

        return image, gps_coords

These transforms are copied from the baseline notebook. Training gets augmentation; validation uses deterministic inference transforms.


In [ ]:
from torchvision.transforms import InterpolationMode

In [ ]:
transform = transforms.Compose([
    transforms.RandomResizedCrop(224),  # Random crop and resize to 224x224
    transforms.RandomHorizontalFlip(),  # Random horizontal flip
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),  # Random color jitter
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Optionally, you can create a separate transform for inference without augmentations
inference_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [ ]:
# Create the training dataset and dataloader
train_dataset = GPSImageDataset(hf_dataset=dataset_train, transform=transform)
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# Retrieve normalization parameters from the training dataset
lat_mean = train_dataset.latitude_mean
lat_std = train_dataset.latitude_std
lon_mean = train_dataset.longitude_mean
lon_std = train_dataset.longitude_std

print("lat_mean:", lat_mean)
print("lat_std:", lat_std)
print("lon_mean:", lon_mean)
print("lon_std:", lon_std)

In [ ]:
# Create the validation dataset and dataloader using training mean and std
val_dataset = GPSImageDataset(
    hf_dataset=dataset_validation,
    transform=inference_transform,
    lat_mean=lat_mean,
    lat_std=lat_std,
    lon_mean=lon_mean,
    lon_std=lon_std
)
val_dataloader = DataLoader(val_dataset, batch_size=32, shuffle=False)


## Batch Shape Check

Confirms the image tensors and normalized GPS labels are the expected format.

In [ ]:
images, gps_coords = next(iter(train_dataloader))

print("image batch shape:", images.shape)
print("gps batch shape:", gps_coords.shape)
print("first normalized gps label:", gps_coords[0])

In [ ]:
# Convert normalized labels back to raw latitude/longitude for inspection.
gps_coords_denorm = gps_coords * torch.tensor([lat_std, lon_std]) + torch.tensor([lat_mean, lon_mean])
gps_coords_denorm[:5]

## Visualize Data

In [ ]:
import matplotlib.pyplot as plt
import torchvision.transforms as transforms
import numpy as np

def denormalize(tensor, mean, std):
    mean = np.array(mean)
    std = np.array(std)
    tensor = tensor.numpy().transpose((1, 2, 0))  # Convert from C x H x W to H x W x C
    tensor = std * tensor + mean  # Denormalize
    tensor = np.clip(tensor, 0, 1)  # Clip to keep pixel values between 0 and 1
    return tensor

data_iter = iter(train_dataloader)
images, gps_coords = next(data_iter)  # Get a batch of images and labels
# Denormalize the first image in the batch for display
itr = 0
for im in images:
  image = denormalize(im, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

  # Plot the image
  plt.imshow(image)
  plt.title(f'Latitude: {gps_coords[itr][0].item():.4f}, Longitude: {gps_coords[itr][1].item():.4f}')
  plt.axis('off')
  plt.show()
  itr += 1

## Next Step

At this point, `train_dataloader` and `val_dataloader` are ready to use for actual model code. Feel free to adjust the transforms or other parts of the pipeline if you think it will make the models behave better in the end!
